# Sri Lankan Vehicle Price Prediction using Ensemble Learning

**Module:** Machine Learning and Pattern Recognition  
**Task:** Used Car Price Prediction and Ensemble Model Comparison (Bagging, Boosting, and Stacking)  
**Dataset:** Sri Lankan Vehicle Dataset (scraped from ikman.lk & Riyasewana.lk)  


### 1. Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    StackingRegressor
)
from sklearn.linear_model import RidgeCV
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

### 2. Load and Explore Data

In [ ]:
df = pd.read_csv('data/car_price_dataset.csv')
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])
df.columns = [c.strip() for c in df.columns]
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

### 3. Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Top 10 brands
top_brands = df['Brand'].value_counts().head(10)
sns.barplot(x=top_brands.values, y=top_brands.index, ax=ax[0])
ax[0].set_title('Top 10 Vehicle Brands in Sri Lanka')
ax[0].set_xlabel('Count')

# Price distribution
sns.histplot(df['Price'], bins=30, kde=True, ax=ax[1])
ax[1].set_title('Price Distribution (Lakhs LKR)')
ax[1].set_xlabel('Price (Lakhs)')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(data=df[df['Price'] <= 200], x='Fuel Type', y='Price')
plt.title('Vehicle Price by Fuel Type (Prices <= 200 Lakhs)')
plt.ylabel('Price in Lakhs')
plt.show()

### 4. Data Cleaning & Feature Engineering

In [ ]:
# Remove noise and unrealistic values
df_clean = df[(df['YOM'] >= 1980) & (df['YOM'] <= 2025)].copy()
df_clean = df_clean[(df_clean['Price'] >= 3.0) & (df_clean['Price'] <= 350.0)]
df_clean = df_clean[(df_clean['Millage(KM)'] >= 1000) & (df_clean['Millage(KM)'] <= 600000)]
df_clean = df_clean[(df_clean['Engine (cc)'] >= 600) & (df_clean['Engine (cc)'] <= 4500)]

# Feature engineering
df_clean['Vehicle_Age'] = (2025 - df_clean['YOM']).clip(lower=0)
df_clean['Km_Per_Year'] = df_clean['Millage(KM)'] / (df_clean['Vehicle_Age'] + 1)

binary_cols = ['AIR CONDITION', 'POWER STEERING', 'POWER MIRROR', 'POWER WINDOW']
for col in binary_cols:
    df_clean[col] = (df_clean[col].str.strip().str.lower() == 'available').astype(int)
df_clean['Feature_Score'] = df_clean[binary_cols].sum(axis=1)

# Group low frequency categories
top_brands = df_clean['Brand'].value_counts().head(20).index.tolist()
df_clean['Brand_Cleaned'] = df_clean['Brand'].apply(lambda x: x if x in top_brands else 'Other')

top_models = df_clean['Model'].value_counts().head(60).index.tolist()
df_clean['Model_Cleaned'] = df_clean['Model'].apply(lambda x: x if x in top_models else 'Other')

top_towns = df_clean['Town'].value_counts().head(30).index.tolist()
df_clean['Town_Cleaned'] = df_clean['Town'].apply(lambda x: x if x in top_towns else 'Other')

print(f"Cleaned shape: {df_clean.shape}")

### 5. Preprocessing & Train-Test Split

In [ ]:
categorical_features = ['Brand_Cleaned', 'Model_Cleaned', 'Gear', 'Fuel Type', 'Town_Cleaned', 'Condition']
numeric_features = ['YOM', 'Vehicle_Age', 'Engine (cc)', 'Millage(KM)', 'Km_Per_Year', 
                    'AIR CONDITION', 'POWER STEERING', 'POWER MIRROR', 'POWER WINDOW', 'Feature_Score']

X = df_clean[categorical_features + numeric_features]
y = df_clean['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

print(f"X_train encoded shape: {X_train_prep.shape}")
print(f"X_test encoded shape: {X_test_prep.shape}")

### 6. Model Training & Evaluation

In [ ]:
def evaluate_model(name, model, paradigm, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
    
    return {
        'Model': name,
        'Paradigm': paradigm,
        'MAE (LKR Lakhs)': round(mae, 2),
        'RMSE (LKR Lakhs)': round(rmse, 2),
        'R2 Score': round(r2, 4),
        'MAPE (%)': round(mape, 2)
    }, model

models_to_train = {
    ('Random Forest', 'Bagging'): RandomForestRegressor(n_estimators=180, max_depth=20, random_state=42, n_jobs=-1),
    ('Extra Trees', 'Bagging'): ExtraTreesRegressor(n_estimators=180, max_depth=20, random_state=42, n_jobs=-1),
    ('Gradient Boosting', 'Boosting'): GradientBoostingRegressor(n_estimators=200, learning_rate=0.08, max_depth=6, random_state=42),
    ('XGBoost', 'Boosting'): XGBRegressor(n_estimators=250, learning_rate=0.06, max_depth=6, random_state=42, n_jobs=-1),
    ('LightGBM', 'Boosting'): LGBMRegressor(n_estimators=250, learning_rate=0.06, num_leaves=31, random_state=42, verbose=-1),
    ('CatBoost', 'Boosting'): CatBoostRegressor(iterations=350, learning_rate=0.08, depth=6, random_seed=42, verbose=0)
}

results = []
trained_models = {}

for (m_name, m_paradigm), m_obj in models_to_train.items():
    print(f"Training {m_name}...")
    res, fitted = evaluate_model(m_name, m_obj, m_paradigm, X_train_prep, y_train, X_test_prep, y_test)
    results.append(res)
    trained_models[m_name] = fitted

### 7. Hybrid Stacking Model

In [ ]:
stacking_reg = StackingRegressor(
    estimators=[
        ('rf', RandomForestRegressor(n_estimators=120, max_depth=16, random_state=42, n_jobs=-1)),
        ('xgb', XGBRegressor(n_estimators=150, learning_rate=0.07, max_depth=6, random_state=42, n_jobs=-1)),
        ('cat', CatBoostRegressor(iterations=200, learning_rate=0.08, depth=6, random_seed=42, verbose=0))
    ],
    final_estimator=RidgeCV(),
    cv=3,
    n_jobs=-1
)

print("Training Stacking Model...")
res_stack, fitted_stack = evaluate_model('Hybrid Stacking (RF+XGB+Cat)', stacking_reg, 'Hybrid Stacking', X_train_prep, y_train, X_test_prep, y_test)
results.append(res_stack)
trained_models['Hybrid Stacking'] = fitted_stack

### 8. Model Comparison and Evaluation Table

In [ ]:
results_df = pd.DataFrame(results).sort_values(by='R2 Score', ascending=False)
results_df

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=results_df, x='R2 Score', y='Model', ax=ax[0])
ax[0].set_xlim(0.80, 0.95)
ax[0].set_title('R2 Score Comparison')

sns.barplot(data=results_df, x='MAE (LKR Lakhs)', y='Model', ax=ax[1])
ax[1].set_title('MAE in Lakhs (Lower is Better)')

plt.tight_layout()
plt.show()

### 9. Observations & Summary
- **LightGBM** reached the best R² score (0.9013) with an RMSE of 13.46 Lakhs.
- **Random Forest** had the lowest MAE of 5.75 Lakhs and lowest MAPE (14.47%).
- **Hybrid Stacking** yielded a solid 0.8881 R² score by taking predictions from Random Forest, XGBoost, and CatBoost.
